# MiniMind-O 01 — 数据检查与 SFT 多阶段训练

本 Notebook 专注于在 Google Colab / Jupyter 中完成 MiniMind-O 的完整训练与权重转换：
1. 环境初始化与 Google Drive 持久化（推荐开启，防止 Colab 断连丢失断点）
2. 下载冻结编码器（SenseVoiceSmall / SigLIP2 / Mimi）、基座语言权重（`llm_768.pth`）与 parquet 数据集
3. 预览 parquet 训练样本（文本对话 + Mimi 解码语音试听）
4. 执行 SFT 训练（T2A -> A2A Projector -> A2A Full -> I2T Vision Projector）并绘制 Loss / LR 曲线
5. 导出 HuggingFace Transformers 格式供推理和 WebUI 使用

In [ ]:
#@title 1. 环境初始化与依赖安装 (Colab / 本地 Jupyter 通用)
import os
import shutil
import subprocess
import sys

REPO_URL = "https://github.com/jingyaogong/minimind-o.git"  #@param {type:"string"}
BRANCH = ""  #@param {type:"string"} # 留空克隆默认分支，或填入特定分支名
INSTALL_DEPS = True  #@param {type:"boolean"}

def _find_repo_root():
    for cand in [os.getcwd(), os.path.dirname(os.getcwd()), "/content/minimind-o"]:
        if os.path.exists(os.path.join(cand, "notebook_utils.py")) and os.path.exists(os.path.join(cand, "model", "model_omni.py")):
            return os.path.abspath(cand)
    return None

root = _find_repo_root()
if root is None:
    target = "/content/minimind-o" if "google.colab" in sys.modules else os.path.abspath("./minimind-o")
    cmd = ["git", "clone", "--depth", "1"]
    if BRANCH.strip():
        cmd += ["-b", BRANCH.strip()]
    cmd += [REPO_URL, target]
    print("Cloning repository:", " ".join(cmd))
    subprocess.run(cmd, check=True)
    root = target

os.chdir(root)
if root not in sys.path:
    sys.path.insert(0, root)

if INSTALL_DEPS:
    req_file = "requirements_colab.txt" if os.path.exists("requirements_colab.txt") else "requirements.txt"
    print(f"Installing dependencies from {req_file} ...")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", req_file], check=True)
    if shutil.which("ffmpeg") is None and "google.colab" in sys.modules:
        subprocess.run(["apt-get", "update", "-qq"], check=False)
        subprocess.run(["apt-get", "install", "-y", "-qq", "ffmpeg"], check=False)

from notebook_utils import check_environment
env_info = check_environment()

In [ ]:
#@title 2. (可选) 挂载 Google Drive 持久化保存权重与断点
USE_GOOGLE_DRIVE = False  #@param {type:"boolean"}
DRIVE_ROOT = "/content/drive/MyDrive/minimind-o"  #@param {type:"string"}

from notebook_utils import setup_persistent_storage

if USE_GOOGLE_DRIVE:
    storage_paths = setup_persistent_storage(drive_root=DRIVE_ROOT, link_out_and_checkpoints=True)
else:
    storage_paths = {
        "save_dir": "./out",
        "checkpoint_dir": "./checkpoints",
        "dataset_dir": "./dataset",
        "hf_dir": "./",
    }
    os.makedirs("./out", exist_ok=True)
    os.makedirs("./checkpoints", exist_ok=True)
    print("Using local workspace directories:", storage_paths)

In [ ]:
#@title 3. 下载编码器、基座权重 (llm_768) 与训练数据集
SOURCE = "huggingface"  #@param ["huggingface", "modelscope"]
DATASET_PRESET = "mini"  #@param ["mini", "full", "all"]
USE_MOE = False  #@param {type:"boolean"}

from notebook_utils import download_encoders, download_pytorch_weights, download_datasets

download_encoders(source=SOURCE, include_campplus=False)
base_weight = "llm_768_moe.pth" if USE_MOE else "llm_768.pth"
download_pytorch_weights(filenames=[base_weight], source=SOURCE, save_dir=storage_paths["save_dir"])
download_datasets(preset=DATASET_PRESET, source=SOURCE, dataset_dir=storage_paths["dataset_dir"])

In [ ]:
#@title 4. 数据集样本检查与语音试听
DATA_FILE = "sft_t2a_mini.parquet"  #@param ["sft_t2a_mini.parquet", "sft_a2a_mini.parquet", "sft_t2a.parquet", "sft_a2a.parquet", "sft_i2t.parquet"]
NUM_SAMPLES = 2  #@param {type:"integer"}

from notebook_utils import inspect_parquet_dataset

data_path = os.path.join(storage_paths["dataset_dir"], DATA_FILE)
if not os.path.exists(data_path):
    data_path = os.path.join("./dataset", DATA_FILE)
inspect_parquet_dataset(data_path, num_samples=NUM_SAMPLES, device=env_info["device"])

In [ ]:
#@title 5. 运行单个训练阶段或自定义配置
STAGE = "stage1_t2a_mini"  #@param ["stage1_t2a_mini", "stage2_a2a_proj_mini", "stage3_a2a_mini", "stage4_i2t_proj", "custom"]
CUSTOM_DATA_FILE = "sft_t2a_mini.parquet"  #@param ["sft_t2a_mini.parquet", "sft_a2a_mini.parquet", "sft_t2a.parquet", "sft_a2a.parquet", "sft_i2t.parquet"]
MODE = "all"  #@param ["all", "audio_proj", "vision_proj"]
EPOCHS = 1  #@param {type:"integer"}
BATCH_SIZE = 16  #@param {type:"integer"}
LEARNING_RATE = 5e-4  #@param {type:"number"}
MAX_SEQ_LEN = 512  #@param {type:"integer"}
FROM_WEIGHT = "llm"  #@param {type:"string"}
SAVE_WEIGHT = "sft_zero"  #@param {type:"string"}
FROM_RESUME = False  #@param {type:"boolean"}
MAX_STEPS = 100  #@param {type:"integer"} # 0=完整Epoch
MAX_SAMPLES = 0  #@param {type:"integer"} # 0=全量数据
LOG_INTERVAL = 10  #@param {type:"integer"}
SAVE_INTERVAL = 500  #@param {type:"integer"}

from trainer.train_sft_omni import OmniTrainConfig, run_training
from notebook_utils import plot_training_history

presets = {
    "stage1_t2a_mini": dict(data_file="sft_t2a_mini.parquet", mode="all", lr=5e-4, max_seq_len=512, from_weight="llm", save_weight="sft_zero"),
    "stage2_a2a_proj_mini": dict(data_file="sft_a2a_mini.parquet", mode="audio_proj", lr=5e-4, max_seq_len=640, from_weight="sft_zero", save_weight="sft_zero"),
    "stage3_a2a_mini": dict(data_file="sft_a2a_mini.parquet", mode="all", lr=2e-5, max_seq_len=768, from_weight="sft_zero", save_weight="sft_zero"),
    "stage4_i2t_proj": dict(data_file="sft_i2t.parquet", mode="vision_proj", lr=5e-5, max_seq_len=768, from_weight="sft_zero", save_weight="sft_zero"),
}

p = presets.get(STAGE, {})
data_name = p.get("data_file", CUSTOM_DATA_FILE) if STAGE != "custom" else CUSTOM_DATA_FILE
data_path = os.path.join(storage_paths["dataset_dir"], data_name)
if not os.path.exists(data_path):
    data_path = os.path.join("./dataset", data_name)

cfg = OmniTrainConfig(
    save_dir=storage_paths["save_dir"],
    checkpoint_dir=storage_paths["checkpoint_dir"],
    save_weight=p.get("save_weight", SAVE_WEIGHT) if STAGE != "custom" else SAVE_WEIGHT,
    from_weight=p.get("from_weight", FROM_WEIGHT) if STAGE != "custom" else FROM_WEIGHT,
    from_resume=int(FROM_RESUME),
    data_path=data_path,
    epochs=EPOCHS,
    batch_size=min(BATCH_SIZE, env_info["recommended_batch_size"]) if env_info["cuda_available"] else 2,
    learning_rate=p.get("lr", LEARNING_RATE) if STAGE != "custom" else LEARNING_RATE,
    max_seq_len=p.get("max_seq_len", MAX_SEQ_LEN) if STAGE != "custom" else MAX_SEQ_LEN,
    mode=p.get("mode", MODE) if STAGE != "custom" else MODE,
    use_moe=int(USE_MOE),
    dtype=env_info["recommended_dtype"],
    num_workers=env_info["recommended_num_workers"],
    use_compile=0,
    log_interval=LOG_INTERVAL,
    save_interval=SAVE_INTERVAL,
    max_steps=MAX_STEPS,
    max_samples=MAX_SAMPLES,
)

history = run_training(cfg)
plot_training_history(history, title=f"MiniMind-O SFT ({STAGE})")

In [ ]:
#@title 6. 导出为 HuggingFace Transformers 格式
TORCH_WEIGHT_NAME = "sft_zero"  #@param {type:"string"}
EXPORT_DIR = "./minimind-3o-trained"  #@param {type:"string"}

from model.model_omni import OmniConfig
from scripts.convert_omni import convert_torch2transformers

lm_cfg = OmniConfig(hidden_size=768, num_hidden_layers=8, use_moe=USE_MOE)
moe_suffix = "_moe" if USE_MOE else ""
torch_path = os.path.join(storage_paths["save_dir"], f"{TORCH_WEIGHT_NAME}_768{moe_suffix}.pth")
out_dtype = torch.bfloat16 if env_info["bf16_supported"] else torch.float16
convert_torch2transformers(torch_path, EXPORT_DIR, lm_cfg, dtype=out_dtype)